# Module 5 · Session 3 (Session 23)
## Containers, portability and reproducibility in HPC
**Bachelor in Computer Science and Artificial Intelligence · Oscar Diez · 2026–27**

Why does yesterday's working simulation fail on another machine? In this session you will investigate the environment around a program, package a small experiment and decide what evidence another researcher needs to repeat it.

**Learning goals**
1. Explain dependencies, modules, package specifications and container boundaries.
2. Run a small scientific calculation with recorded inputs, code and parameters.
3. Distinguish exact repeatability from numerical agreement and performance portability.
4. Explain Apptainer bind mounts, Slurm launch patterns and MPI/GPU compatibility.
5. Build and check a small reproducibility package and a workflow cache.

**80-minute route:** setup and dependencies (0–15), modules and packages (15–25), containers (25–40), reproducible experiment (40–55), cluster integration (55–65), workflows and handover (65–80).

Run cells in order. Predict the result before each demo, then explain what you observed. The core needs **Python 3.9+ and the standard library only**, without internet, root access or a GPU. Optional cluster cells require an existing allocation and a compatible local image. They print a clear skip when unavailable. This notebook never submits jobs or installs software automatically.

The older `chapters_examples/23.M5.S3.Hybrid_Computing(5_3).ipynb` covers a different topic. This is the syllabus-aligned Session 23 notebook.


## 0 · Environment and MCQ setup
A notebook kernel is an already-running process. Loading a module in a separate terminal cannot retroactively change its interpreter.

All generated files go inside `m5s3_repro_lab/` in your current working directory. Rerunning cells replaces only the lab's named files. `answers` records your MCQs locally. Feedback appears after you choose an answer.


In [1]:
import os, sys, json, math, random, hashlib, platform, shutil, subprocess, csv, time, venv
from pathlib import Path
ROOT = Path("m5s3_repro_lab").resolve()
ROOT.mkdir(exist_ok=True)
QUIZ, answers = {}, {}
def check(number):
    chosen = answers.get(number)
    if chosen is None:
        print(f"Q{number}: choose an answer before checking.")
        return
    chosen = str(chosen).strip().upper()
    if chosen not in "ABCD" or len(chosen) != 1:
        print("Use one letter: A, B, C or D.")
        return
    item = QUIZ[number]
    print("Correct." if chosen == item["correct"] else "Try again after reading the explanation.")
    print(item["explanation"])
def run(args, **kw):
    result = subprocess.run([str(a) for a in args], text=True, capture_output=True,
                            timeout=30, **kw)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout.strip()
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Platform:", platform.system(), platform.machine())
print("Apptainer available:", bool(shutil.which("apptainer")))
print("Slurm job:", os.environ.get("SLURM_JOB_ID", "none"))
print("Lab folder:", ROOT)


Python: 3.12.14
Interpreter: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Platform: Linux x86_64
Apptainer available: False
Slurm job: none
Lab folder: /workspace/scratch/9fc41d5dd068/m5s3_repro_lab


## 1 · A dependency can change the result
**Predict:** both programs below run the same application text. Will the answers match?

We deliberately create two tiny versions of a teaching library. Version 1 interprets the input as Celsius. Version 2 mistakenly interprets it as Fahrenheit. These are local teaching examples, not a real package release history.


In [2]:
app = "import thermo; print(round(thermo.to_kelvin(20), 3))"
versions = {
    "v1": "def to_kelvin(x): return x + 273.15\n",
    "v2": "def to_kelvin(x): return (x - 32) * 5 / 9 + 273.15\n",
}
observed = {}
for name, source in versions.items():
    folder = ROOT / name
    folder.mkdir(exist_ok=True)
    (folder / "thermo.py").write_text(source, encoding="utf-8")
    observed[name] = run([sys.executable, "-c", app], cwd=folder)
    print(name, "->", observed[name])
assert observed["v1"] == "293.15"
assert observed["v1"] != observed["v2"]


v1 -> 293.15
v2 -> 266.483


**Read the result:** the application text stayed the same, yet a dependency changed the interpretation. Saving only the notebook would miss this difference. Other failures include missing libraries, incompatible shared-library interfaces, CPU instruction sets and GPU drivers.


### Checkpoint Q1
The same application gives different temperatures after a dependency change. What should you record?

- **A.** Only the final answer
- **B.** Application code and the exact dependency environment
- **C.** Only the machine name
- **D.** Only the elapsed time

Change `None` to your chosen letter and run the next cell. You can retry.


In [3]:
QUIZ[1] = {"correct": 'B', "explanation": 'Code and dependency versions both affect execution. An input unit or API change can alter the answer.'}
answers[1] = None  # replace with "A", "B", "C" or "D"
check(1)


Q1: choose an answer before checking.


## 2 · Environment variables and modules
A module selects software already installed by the site. It typically changes shell variables such as `PATH` and library search paths. It does not copy that software into your project.

**Predict:** what will the child process print before and after we set an environment variable? This small example demonstrates process inheritance, not an implementation of a module system.


In [4]:
child = "import os; print(os.environ.get('M5S3_EXAMPLE_SETTING', 'unset'))"
clean = dict(os.environ)
clean.pop("M5S3_EXAMPLE_SETTING", None)
selected = dict(clean, M5S3_EXAMPLE_SETTING="two-threads")
print("child without setting:", run([sys.executable, "-c", child], env=clean))
print("child with setting:", run([sys.executable, "-c", child], env=selected))
print("parent setting:", os.environ.get("M5S3_EXAMPLE_SETTING", "unset"))


child without setting: unset
child with setting: two-threads
parent setting: unset


On a cluster terminal, inspect the software before changing it:
```bash
module avail
module list
module show <site-module-name>
module load <site-module-name>
which python3
python3 --version
```
`<site-module-name>` is a placeholder. Use names from your cluster. The `module` command is often a shell function, so `shutil.which("module")` is not a reliable availability test. In a batch job load the required modules in that job's shell. To select a different notebook interpreter, use the site's kernel procedure.


### Checkpoint Q2
You load a Python module in a terminal while a notebook kernel is running. What follows?

- **A.** The running kernel automatically changes Python
- **B.** The notebook loses its input data
- **C.** The terminal environment changes, while the existing kernel keeps its interpreter
- **D.** The module creates a container

Change `None` to your chosen letter and run the next cell. You can retry.


In [5]:
QUIZ[2] = {"correct": 'C', "explanation": 'Environment changes affect that shell and future children. An existing kernel continues using its original interpreter.'}
answers[2] = None  # replace with "A", "B", "C" or "D"
check(2)


Q2: choose an answer before checking.


## 3 · A Python environment and a package specification
A virtual environment separates Python package locations. It still relies on the base interpreter, operating system and native libraries.

The next cell creates an empty environment **without downloading pip or packages**, then checks the interpreter. This is a real environment, not a simulation. It can take a few seconds.


In [6]:
env_dir = ROOT / "empty_venv"
venv.EnvBuilder(with_pip=False).create(env_dir)
venv_python = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print(run([venv_python, "-c",
           "import sys; print('inside venv:', sys.prefix != sys.base_prefix); print(sys.version.split()[0])"]))
print("Packages were not installed.")


inside venv: True
3.12.14
Packages were not installed.


For Python applications, a package list or lockfile helps describe dependencies. `pip freeze` describes Python distributions visible to an interpreter, but does not capture GPU drivers, CPU architecture or all system libraries.

Spack manages compiled scientific software and compiler/build variants. A `spack.yaml` states requirements. Concretization resolves a full dependency graph and writes `spack.lock`. Save both, alongside the Spack version and site/compiler configuration. A lockfile alone does not guarantee a working binary on every architecture.

The following cell writes a small **example specification**. It does not install MPI or invent a lockfile.


In [7]:
spack_spec = """spack:
  specs:
    - zlib
  concretizer:
    unify: true
"""
(ROOT / "spack.yaml").write_text(spack_spec, encoding="utf-8")
print(spack_spec)
print("On a prepared system: spack env create m5s3 spack.yaml")
print("Then: spack env activate m5s3; spack concretize; spack install")
print("Save the generated spack.lock. These commands have not run here.")


spack:
  specs:
    - zlib
  concretizer:
    unify: true

On a prepared system: spack env create m5s3 spack.yaml
Then: spack env activate m5s3; spack concretize; spack install
Save the generated spack.lock. These commands have not run here.


### Checkpoint Q3
What does Spack concretization add to an abstract package specification?

- **A.** A resolved dependency graph with concrete build choices
- **B.** A GPU allocation
- **C.** A guarantee of equal runtime on all CPUs
- **D.** An automatic scientific validation

Change `None` to your chosen letter and run the next cell. You can retry.


In [8]:
QUIZ[3] = {"correct": 'A', "explanation": 'Concretization resolves dependency versions and build choices. Hardware compatibility and scientific checks remain separate tasks.'}
answers[3] = None  # replace with "A", "B", "C" or "D"
check(3)


Q3: choose an answer before checking.


## 4 · Container boundaries
A VM includes a guest kernel. A Linux application container shares the host kernel and packages user-space files. Apptainer supports scientific workloads and SIF images. Standard execution normally uses the calling user's identity and permissions. Site configuration still matters.

A container helps preserve libraries and runtimes. It does not carry a new physical GPU, automatically allocate resources or make arbitrary host MPI and drivers compatible. Apptainer and SingularityCE are related projects with different development histories. Check the runtime available at your site.

**Pair discussion:** which of these belong in the image, and which must the cluster provide: Python libraries, application code, host kernel, scheduler allocation, GPU driver?


### Checkpoint Q4
Which resource does a normal Linux application container share with the host?

- **A.** An independent guest kernel
- **B.** The host kernel
- **C.** A guaranteed number of GPU devices
- **D.** A private physical network adapter

Change `None` to your chosen letter and run the next cell. You can retry.


In [9]:
QUIZ[4] = {"correct": 'B', "explanation": 'Application containers share the host kernel. They still need suitable host hardware and runtime integration.'}
answers[4] = None  # replace with "A", "B", "C" or "D"
check(4)


Q4: choose an answer before checking.


## 5 · One recorded scientific experiment
We use a small temperature dataset throughout the remaining examples. The input is generated with a local seeded generator and then saved as a CSV artifact. Hashing the saved bytes gives a precise input identity.

**Predict:** what changes if we alter the threshold but keep the input bytes unchanged?


In [10]:
rng = random.Random(23)
input_file = ROOT / "temperatures.csv"
with input_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["temperature_c"])
    writer.writerows([[f"{rng.uniform(10, 30):.6f}"] for _ in range(1000)])
print("input rows:", 1000)
print("input SHA256:", sha(input_file))


input rows: 1000
input SHA256: 4919f62bd3a309994feb76c8baec9bc97e12e12a55737945ef6d5ca2a2f20caa


### The portable payload
This script uses only the Python standard library. It computes the mean temperature and the fraction above a threshold. It writes the scientific result separately from provenance metadata.

Read the script before running it. Locate the input, parameter, code hash and input hash. `math.fsum` improves accumulation accuracy, but does not make every numerical application identical across machines.


In [11]:
payload = r'''import argparse, csv, json, hashlib, math, platform, sys, os
from pathlib import Path
p = argparse.ArgumentParser()
p.add_argument("--input", required=True)
p.add_argument("--threshold", type=float, default=25.0)
p.add_argument("--output", required=True)
a = p.parse_args()
input_path = Path(a.input)
with input_path.open(newline="", encoding="utf-8") as f:
    values = [float(row["temperature_c"]) for row in csv.DictReader(f)]
if not values:
    raise ValueError("input must contain temperatures")
result = {"count": len(values), "mean_c": math.fsum(values) / len(values),
          "fraction_above": sum(x > a.threshold for x in values) / len(values)}
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
record = {"result": result, "parameters": {"threshold_c": a.threshold},
          "input_sha256": sha(input_path), "code_sha256": sha(__file__),
          "runtime": {"python": sys.version.split()[0],
                      "platform": platform.system(), "machine": platform.machine(),
                      "executable": sys.executable,
                      "omp_threads": os.environ.get("OMP_NUM_THREADS", "unset")}}
Path(a.output).write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result, sort_keys=True))
'''
script = ROOT / "analysis.py"
script.write_text(payload, encoding="utf-8")
print("Wrote", script.name)
print("Code SHA256:", sha(script))


Wrote analysis.py
Code SHA256: 420c82669a5bfee549544893822609a937e11739598b21259e6ae7bac20d2ae7


In [12]:
def host_run(threshold, name):
    output = ROOT / name
    print(run([sys.executable, script, "--input", input_file,
               "--threshold", threshold, "--output", output]))
    return json.loads(output.read_text())
first = host_run(25, "host_25_a.json")
second = host_run(25, "host_25_b.json")
changed = host_run(20, "host_20.json")
assert first["result"] == second["result"]
assert first["input_sha256"] == changed["input_sha256"]
assert first["result"]["fraction_above"] != changed["result"]["fraction_above"]
print("Repeated result identical:", first["result"] == second["result"])
print("Changed threshold changes fraction:", changed["result"]["fraction_above"])
print(json.dumps(first, indent=2))


{"count": 1000, "fraction_above": 0.253, "mean_c": 20.035621014}
{"count": 1000, "fraction_above": 0.253, "mean_c": 20.035621014}
{"count": 1000, "fraction_above": 0.511, "mean_c": 20.035621014}
Repeated result identical: True
Changed threshold changes fraction: 0.511
{
  "result": {
    "count": 1000,
    "mean_c": 20.035621014,
    "fraction_above": 0.253
  },
  "parameters": {
    "threshold_c": 25.0
  },
  "input_sha256": "4919f62bd3a309994feb76c8baec9bc97e12e12a55737945ef6d5ca2a2f20caa",
  "code_sha256": "420c82669a5bfee549544893822609a937e11739598b21259e6ae7bac20d2ae7",
  "runtime": {
    "python": "3.12.14",
    "platform": "Linux",
    "machine": "x86_64",
    "executable": "/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3",
    "omp_threads": "unset"
  }
}


**Explain:** both thresholds see the same input hash and the same mean. The fraction above the threshold changes because the parameter changes. The record lets you separate a scientific parameter change from an input change.

**Try it:** run `host_run(22, "host_22.json")`. Predict whether the mean should change. Then edit one input row, rerun and compare the input hash. Rerun the input-generation cell to restore the original dataset.


### Checkpoint Q5
The image is unchanged, but a new threshold gives a new fraction. What explains this?

- **A.** The container is corrupted
- **B.** The CPU changed the input file
- **C.** The scientific parameter is part of the experiment
- **D.** An input hash records every parameter automatically

Change `None` to your chosen letter and run the next cell. You can retry.


In [13]:
QUIZ[5] = {"correct": 'C', "explanation": 'The experiment includes parameters as well as code, environment and data. The input hash alone does not record the threshold.'}
answers[5] = None  # replace with "A", "B", "C" or "D"
check(5)


Q5: choose an answer before checking.


## 6 · Random seeds and floating-point order
A seed helps repeat a pseudorandom sequence within a chosen implementation. It does not guarantee equality across arbitrary libraries, hardware, GPU kernels or parallel reduction orders.

**Predict:** will the two sums below match? Think about the small value added next to a very large one.


In [14]:
a, b = random.Random(7), random.Random(7)
print("Same seeded sequence:", [a.random() for _ in range(4)] == [b.random() for _ in range(4)])
x = [1e16, 1.0, -1e16]
y = [1e16, -1e16, 1.0]
print("ordinary sum, order 1:", sum(x))
print("ordinary sum, order 2:", sum(y))
print("math.fsum:", math.fsum(x), math.fsum(y))
print("close means, chosen tolerance:", math.isclose(first["result"]["mean_c"],
      second["result"]["mean_c"], rel_tol=1e-12, abs_tol=1e-12))


Same seeded sequence: True
ordinary sum, order 1: 1.0
ordinary sum, order 2: 1.0
math.fsum: 1.0 1.0
close means, chosen tolerance: True


Python's ordinary `sum` implementation can change between versions, so inspect your actual output. The cancellation example often yields 0.0 versus 1.0. `math.fsum` yields 1.0 for both here.

For a scientific comparison choose a tolerance from the application's units, numerical method and expected error. A fixed `1e-12` is only a teaching example. Record hardware, resource layout and timing protocol when comparing performance. Identical numerical results do not imply identical runtime.


### Checkpoint Q6
Two parallel reductions differ by a small amount. Which check is strongest?

- **A.** Reject every nonidentical byte
- **B.** Accept any difference if both jobs finish
- **C.** Compare within a scientifically justified tolerance and inspect the run metadata
- **D.** Increase the seed until values match

Change `None` to your chosen letter and run the next cell. You can retry.


In [15]:
QUIZ[6] = {"correct": 'C', "explanation": 'Floating-point order can change numerical results. The tolerance needs scientific justification and provenance helps investigate differences.'}
answers[6] = None  # replace with "A", "B", "C" or "D"
check(6)


Q6: choose an answer before checking.


## 7 · Real Apptainer execution (optional cluster section)
`apptainer exec IMAGE command` runs a command inside an existing image. A bind mount maps a host path into the container. `--cleanenv` reduces inherited host variables, but does not isolate every file, device or host dependency.

Use a **local, trusted image with Python 3** and a compatible architecture. Set `M5S3_IMAGE` to its absolute path, then set `RUN_CONTAINER = True` below. No download occurs automatically. The default path prints a skip.

For a teacher-prepared image, this is an illustrative pull command:
```bash
apptainer pull python311.sif docker://python:3.11-slim
sha256sum python311.sif
```
The tag can move. For a repeatable build resolve and record an OCI digest, use it in the recipe and archive the resulting SIF with its checksum. Verify image provenance separately. A checksum shows identity, not trustworthiness.

**Predict:** why can the same script see `/work/temperatures.csv` when that path exists only inside the container?


In [16]:
RUN_CONTAINER = False  # set True only inside an appropriate cluster allocation
IMAGE = Path(os.environ.get("M5S3_IMAGE", str(ROOT / "python311.sif")))
container_record = None
if not RUN_CONTAINER:
    print("SKIP: optional Apptainer execution is disabled.")
elif not shutil.which("apptainer"):
    print("SKIP: Apptainer is not available in this kernel environment.")
elif not os.environ.get("SLURM_JOB_ID"):
    print("SKIP: obtain a Slurm compute allocation before enabling this cluster exercise.")
elif not IMAGE.is_file():
    print("SKIP: set M5S3_IMAGE to a local Python image.")
else:
    command = ["apptainer", "exec", "--cleanenv", "--bind", str(ROOT)+":/work",
               "--pwd", "/work", IMAGE, "python3", "analysis.py",
               "--input", "temperatures.csv", "--threshold", "25",
               "--output", "container_25.json"]
    print(run(command))
    container_record = json.loads((ROOT / "container_25.json").read_text())
    assert container_record["input_sha256"] == first["input_sha256"]
    assert container_record["result"]["count"] == first["result"]["count"]
    for key in ("mean_c", "fraction_above"):
        assert math.isclose(container_record["result"][key], first["result"][key],
                            rel_tol=1e-12, abs_tol=1e-12)
    print("PASS: container result agrees within the chosen tolerance.")
    print("Image SHA256:", sha(IMAGE))


SKIP: optional Apptainer execution is disabled.


### Checkpoint Q7
What does `--bind /host/project:/work` do?

- **A.** Copies the host kernel into the image
- **B.** Maps an existing host directory to a path in the container
- **C.** Changes an MPI program into OpenMP
- **D.** Makes the image writable permanently

Change `None` to your chosen letter and run the next cell. You can retry.


In [17]:
QUIZ[7] = {"correct": 'B', "explanation": 'A bind mount exposes an existing host directory at a container path. Writes to that mount affect the host files if permissions allow.'}
answers[7] = None  # replace with "A", "B", "C" or "D"
check(7)


Q7: choose an answer before checking.


### A definition file describes construction
This cell writes a teaching recipe. The base tag is a convenient example and is **not immutable**. Before a long-lived scientific release, replace it with a resolved digest and archive the build inputs. Building can require fakeroot, user namespaces, a remote builder or site support. Running a prepared image normally does not require you to be root.


In [18]:
definition = """Bootstrap: docker
From: python:3.11-slim

%labels
    Course HPC_M5S3

%test
    python3 -c "import csv, json, math"

%runscript
    exec python3 "$@"
"""
(ROOT / "experiment.def").write_text(definition, encoding="utf-8")
print(definition)
print("Build separately on an approved build host: apptainer build experiment.sif experiment.def")


Bootstrap: docker
From: python:3.11-slim

%labels
    Course HPC_M5S3

%test
    python3 -c "import csv, json, math"

%runscript
    exec python3 "$@"

Build separately on an approved build host: apptainer build experiment.sif experiment.def


### Checkpoint Q8
What does a SIF checksum establish?

- **A.** The image is safe because it has a hash
- **B.** The exact image bytes match a recorded identity
- **C.** The result is scientifically valid
- **D.** The image runs on every CPU

Change `None` to your chosen letter and run the next cell. You can retry.


In [19]:
QUIZ[8] = {"correct": 'B', "explanation": 'A checksum establishes byte identity. Image trust, compatibility and scientific validity require separate checks.'}
answers[8] = None  # replace with "A", "B", "C" or "D"
check(8)


Q8: choose an answer before checking.


## 8 · Slurm, MPI and GPUs
Slurm allocates resources. Apptainer supplies an application environment inside that allocation. The following script runs **one serial Python task**. Requesting two CPUs does not make this serial script use two threads.

The cell writes a template and does not submit it. Add your site's account and partition options if required. `M5S3_IMAGE` must be an absolute image path. Submit from the lab folder after reviewing it.


In [20]:
batch = r'''#!/bin/bash
#SBATCH --job-name=m5s3-repro
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=2
#SBATCH --time=00:05:00
#SBATCH --output=m5s3-%j.out
set -eu
# Load the site's Apptainer module here if required.
: "${M5S3_IMAGE:?export an absolute path to a local Python SIF before sbatch}"
cd "$SLURM_SUBMIT_DIR"
srun apptainer exec --cleanenv   --bind "$PWD:/work" --pwd /work "$M5S3_IMAGE"   python3 analysis.py --input temperatures.csv --threshold 25 --output slurm_result.json
'''
(ROOT / "run.slurm").write_text(batch, encoding="utf-8")
print(batch)
print("Not submitted. Review, then from the lab folder run: sbatch run.slurm")


#!/bin/bash
#SBATCH --job-name=m5s3-repro
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=2
#SBATCH --time=00:05:00
#SBATCH --output=m5s3-%j.out
set -eu
# Load the site's Apptainer module here if required.
: "${M5S3_IMAGE:?export an absolute path to a local Python SIF before sbatch}"
cd "$SLURM_SUBMIT_DIR"
srun apptainer exec --cleanenv   --bind "$PWD:/work" --pwd /work "$M5S3_IMAGE"   python3 analysis.py --input temperatures.csv --threshold 25 --output slurm_result.json

Not submitted. Review, then from the lab folder run: sbatch run.slurm


For the MPI + OpenMP program from Session 2, a **site-approved** pattern is:
```bash
export OMP_NUM_THREADS="$SLURM_CPUS_PER_TASK"
export APPTAINERENV_OMP_NUM_THREADS="$OMP_NUM_THREADS"
srun apptainer exec --cleanenv image.sif /opt/app/hybrid
```
Use the rank count and CPUs per task requested in the batch script. Some sites require particular `srun --mpi` options or a host `mpirun`. Check the host/container MPI ABI, process manager and interconnect libraries. Do not nest an unrelated `mpirun` inside every Slurm task.

For NVIDIA GPU applications the familiar starting option is `apptainer exec --nv ...`. It exposes suitable host devices and libraries; you still need an allocated GPU and a compatible host driver/application stack. The notebook does not request a GPU or assume a partition name.


### Checkpoint Q9
What does adding `--nv` to an Apptainer command guarantee?

- **A.** A free GPU allocation
- **B.** Identical speed on all GPUs
- **C.** Automatic compatibility with every CUDA application
- **D.** None of those guarantees

Change `None` to your chosen letter and run the next cell. You can retry.


In [21]:
QUIZ[9] = {"correct": 'D', "explanation": 'The option exposes NVIDIA support. Allocation, device availability and compatible drivers/application libraries still matter.'}
answers[9] = None  # replace with "A", "B", "C" or "D"
check(9)


Q9: choose an answer before checking.


### Checkpoint Q10
An MPI binary starts locally but fails across two nodes in the container. What should you investigate first?

- **A.** Host/container MPI, process manager and interconnect compatibility
- **B.** The colour of the terminal
- **C.** Whether the image has a shorter filename
- **D.** Whether the code has more comments

Change `None` to your chosen letter and run the next cell. You can retry.


In [22]:
QUIZ[10] = {"correct": 'A', "explanation": 'Cross-node startup depends on the site launcher and compatible MPI and network integration, beyond the application files in the image.'}
answers[10] = None  # replace with "A", "B", "C" or "D"
check(10)


Q10: choose an answer before checking.


## 9 · A workflow must invalidate stale results
A workflow records dependencies between steps. Real workflow tools such as Snakemake and Nextflow coordinate tasks, files and execution environments.

We now run a **small teaching cache**, not an implementation of either tool. Its key includes input bytes, code, parameters and the recorded runtime identity. We also verify the output hash. A changed threshold must cause recomputation.

**Predict:** which of the three calls should compute a new result?


In [23]:
cache_file = ROOT / "cache_index.json"
def cached_run(threshold):
    identity = {"input": sha(input_file), "code": sha(script),
                "parameters": {"threshold_c": threshold},
                "environment": {"python": sys.version, "machine": platform.machine(),
                                "executable": sys.executable}}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    index = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    output = ROOT / ("cached_"+key+".json")
    if key in index and output.exists() and sha(output) == index[key]["sha256"]:
        print("REUSE:", threshold)
        return json.loads(output.read_text())
    result = host_run(threshold, output.name)
    index[key] = {"sha256": sha(output), "file": output.name}
    cache_file.write_text(json.dumps(index, indent=2), encoding="utf-8")
    print("COMPUTE:", threshold)
    return result
# Reset this demo's index so the observed sequence is always compute / reuse / compute.
cache_file.write_text("{}", encoding="utf-8")
r1 = cached_run(25)
r2 = cached_run(25)
r3 = cached_run(20)
assert r1["result"] == r2["result"]
assert r1["result"]["fraction_above"] != r3["result"]["fraction_above"]


{"count": 1000, "fraction_above": 0.253, "mean_c": 20.035621014}
COMPUTE: 25
REUSE: 25
{"count": 1000, "fraction_above": 0.511, "mean_c": 20.035621014}
COMPUTE: 20


**Read the result:** the second call reuses an unchanged result. The third call changes a parameter and recomputes. Real scientific workflows also need complete environment identities, external service versions, reference datasets and any other dependencies that affect results. Here the payload has no external Python packages and the environment identity is deliberately small.

**Try it:** change one input value or add a comment to `analysis.py` and call `cached_run(25)` again. Even a comment changes this cache's code hash and triggers recomputation. Explain why a byte-level dependency check can be conservative.


### Checkpoint Q11
A cached result used threshold 25. A new run asks for threshold 20. What should happen?

- **A.** Reuse it because the input filename is unchanged
- **B.** Recompute because the parameters changed
- **C.** Rename the old result to hide the difference
- **D.** Remove all metadata

Change `None` to your chosen letter and run the next cell. You can retry.


In [24]:
QUIZ[11] = {"correct": 'B', "explanation": 'A parameter change changes the experiment identity. A valid dependency key prevents stale result reuse.'}
answers[11] = None  # replace with "A", "B", "C" or "D"
check(11)


Q11: choose an answer before checking.


## 10 · Diagnose a broken handover
Work with a partner. For each symptom identify one likely cause, one check and one fix.

| Symptom | Useful first check |
|---|---|
| `ModuleNotFoundError` | Kernel/interpreter and package environment |
| Shared library cannot load | Native dependency and loader search path |
| `Illegal instruction` | Binary CPU target versus available architecture |
| Input path missing in container | Bind source, destination and working directory |
| MPI starts on one node only | Site launcher, MPI ABI and network integration |
| GPU invisible | Allocation, driver visibility and compatible image |
| Similar result, different time | Hardware, resource layout, data placement and timing method |

Use the standard-library demo to produce a portable handover. A classmate should be able to reproduce the result without your notebook state.


In [25]:
# Package the current threshold-25 run and explicit file identities.
final_record = host_run(25, "handover_result.json")
files = ["analysis.py", "temperatures.csv", "experiment.def", "spack.yaml", "run.slurm", "handover_result.json"]
manifest = {"files": {name: sha(ROOT / name) for name in files},
            "command": "python3 analysis.py --input temperatures.csv --threshold 25 --output reproduced.json",
            "numerical_check": {"rel_tol": 1e-12, "abs_tol": 1e-12,
                                "note": "Teaching tolerance; justify for your scientific application."},
            "container_tested": container_record is not None,
            "image_sha256": sha(IMAGE) if container_record is not None else None}
(ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
readme = """M5S3 reproducibility exercise
1. Use Python 3.9+ with the standard library.
2. Run the command recorded in manifest.json from this directory.
3. Compare input/code hashes and result fields against handover_result.json.
4. Check mean/fraction with the documented teaching tolerance.
5. For containers: review experiment.def and use an archived trusted image.
   The base tag is illustrative and must be pinned for an archived release.
6. For Slurm: review run.slurm and use site allocation instructions.
No container/MPI/GPU success is implied when container_tested is false.
"""
(ROOT / "README.txt").write_text(readme, encoding="utf-8")
for name, expected in manifest["files"].items():
    assert sha(ROOT / name) == expected
print("PASS: manifest file hashes match.")
print("Container tested:", manifest["container_tested"])
print("Handover files:", ", ".join(files + ["manifest.json", "README.txt"]))


{"count": 1000, "fraction_above": 0.253, "mean_c": 20.035621014}
PASS: manifest file hashes match.
Container tested: False
Handover files: analysis.py, temperatures.csv, experiment.def, spack.yaml, run.slurm, handover_result.json, manifest.json, README.txt


### Checkpoint Q12
Which handover best supports scientific reproducibility?

- **A.** A screenshot of the final number
- **B.** The image filename and no data
- **C.** Code, input identity, environment identity, parameters, run instructions and validation criteria
- **D.** The runtime only

Change `None` to your chosen letter and run the next cell. You can retry.


In [26]:
QUIZ[12] = {"correct": 'C', "explanation": 'A complete handover makes the experiment and its checks explicit. Screenshots and filenames alone cannot recover it.'}
answers[12] = None  # replace with "A", "B", "C" or "D"
check(12)


Q12: choose an answer before checking.


### Your short deliverable
1. Choose threshold 22 and save a run record.
2. Explain which fields changed relative to threshold 25 and why.
3. Identify the kernel's Python version and one environment limitation.
4. Give one example where a container still depends on the host.
5. Explain one cache invalidation and one reason to use a numerical tolerance.
6. Save this notebook with your MCQ choices and output. Share the generated lab folder with your partner.

**Optional extension:** repeat inside Apptainer in a compute allocation. Compare result fields, image identity and runtime metadata. Record a skip if you lack the image or allocation. Do not invent a successful run.


In [27]:
attempted = [n for n in QUIZ if isinstance(answers.get(n), str) and answers[n].upper() in "ABCD"]
score = sum(answers[n].upper() == QUIZ[n]["correct"] for n in attempted)
print(f"Attempted {len(attempted)} / {len(QUIZ)} questions. Correct: {score}.")
print("Unanswered:", [n for n in QUIZ if n not in attempted])
(ROOT / "my_mcq_answers.json").write_text(json.dumps(answers, indent=2), encoding="utf-8")
print("Saved local MCQ choices. Save the notebook too.")


Attempted 0 / 12 questions. Correct: 0.
Unanswered: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Saved local MCQ choices. Save the notebook too.


## Reading and command references
**Course reading:** Oscar Diez, *High Performance Computing: Applications and Frontiers*, Chapter 1, sections on modern programming and reproducible science toolchains. Robert Robey and Yuliana Zamora, *Parallel and High Performance Computing* (2021), Chapter 17, “Tools and resources for better code”. These match the Session 23 syllabus.

Official command references (consulted 17 October 2026):
- [Environment Modules](https://modules.readthedocs.io/en/stable/module.html)
- [Spack environments](https://spack.readthedocs.io/en/latest/environments.html)
- [Apptainer exec](https://apptainer.org/docs/user/main/cli/apptainer_exec.html)
- [Apptainer bind paths](https://apptainer.org/docs/user/main/bind_paths_and_mounts.html)
- [Apptainer MPI integration](https://apptainer.org/docs/user/main/mpi.html)
- [Apptainer GPU support](https://apptainer.org/docs/user/main/gpu.html)
- [Slurm sbatch](https://slurm.schedmd.com/sbatch.html)
- [Snakemake documentation](https://snakemake.readthedocs.io/)
- [Nextflow documentation](https://www.nextflow.io/docs/latest/)

**Carry into the project workshop:** another student should be able to explain what ran, reproduce the scientific check and identify what still depends on the cluster.
